# Лабораторная работа №6
## Сканирование параметра β в SIR-модели на сети Петри

В данном скрипте исследуется чувствительность SIR-модели к изменению
коэффициента заражения `β`.

В отличие от базового прогона, где используется один фиксированный набор
параметров, здесь выполняется серия детерминированных симуляций для разных
значений `β`.

Для каждого значения коэффициента заражения вычисляются два итоговых
показателя:

- `peak_I` — максимальное число инфицированных;
- `final_R` — конечное число выздоровевших.

Полученные результаты сохраняются в таблицу и визуализируются на одном
итоговом графике.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение модели

Основная реализация SIR-модели в подходе сетей Петри вынесена в файл
`src/SIRPetri.jl`.

В этом файле определены:

- построение сети Петри;
- детерминированная симуляция;
- стохастическая симуляция;
- построение графиков.

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

using DataFrames, CSV, Plots

## Диапазон значений β

В скрипте задаётся диапазон коэффициента заражения `β`.
Коэффициент выздоровления `γ` фиксируется.

Таким образом, исследуется влияние только одного параметра — скорости
заражения.

In [ ]:
β_range = 0.1:0.05:0.8
γ_fixed = 0.1
tmax = 100.0

## Подготовка массива результатов

Для каждого значения `β` в этот массив будет добавляться запись с тремя
значениями:

- текущее значение `β`;
- максимальное число инфицированных `peak_I`;
- конечное число выздоровевших `final_R`.

In [ ]:
results = []

## Основной цикл сканирования

Для каждого значения коэффициента заражения выполняются следующие шаги:

1. строится сеть Петри модели SIR;
2. запускается детерминированная симуляция;
3. вычисляется пик инфицированных;
4. вычисляется конечное число выздоровевших;
5. результат добавляется в сводную таблицу.

In [ ]:
for β in β_range
    net, u0, _ = build_sir_network(β, γ_fixed)

    df = simulate_deterministic(
        net,
        u0,
        (0.0, tmax),
        saveat = 0.5,
        rates = [β, γ_fixed],
    )

    peak_I = maximum(df.I)
    final_R = df.R[end]

    push!(results, (β = β, peak_I = peak_I, final_R = final_R))
end

## Формирование таблицы результатов

После завершения цикла массив результатов преобразуется в `DataFrame`.

Таблица содержит три колонки:

- `β`;
- `peak_I`;
- `final_R`.

In [ ]:
df_scan = DataFrame(results)

CSV.write(datadir("sir_scan.csv"), df_scan)

## Построение итогового графика

На одном графике отображаются две зависимости:

- `Peak I` — максимальное число инфицированных от `β`;
- `Final R` — конечное число выздоровевших от `β`.

Такой график позволяет увидеть, как увеличение коэффициента заражения
влияет на масштаб эпидемического процесса.

In [ ]:
p = plot(
    df_scan.β,
    [df_scan.peak_I df_scan.final_R],
    label = ["Peak I" "Final R"],
    marker = :circle,
    xlabel = "β (infection rate)",
    ylabel = "Population",
)

savefig(plotsdir("sir_scan.png"))

## Итог

После выполнения скрипта создаются два файла:

- `data/sir_scan.csv` — таблица результатов сканирования;
- `plots/sir_scan.png` — график зависимости `peak_I` и `final_R` от `β`.

Скрипт показывает, как изменение скорости заражения влияет на пик
инфицированных и итоговое число выздоровевших.

In [ ]:
println("Сканирование β завершено. Результат в data/sir_scan.csv")